# Tugas Mandiri: Implementasi Fase-Fase Kompilator

**Mata Kuliah:** Teknik Kompilasi  
---

## Identitas Mahasiswa

| **Nama** | Hizkia Siallagan |
|----------|------------------|
| **NIM**  | 231011400154     |
| **Kelas**| 06TPLMP002       |

---

## Deskripsi Tugas

Berdasarkan materi yang telah dipelajari mengenai *Lexical Analysis*, *EBNF*, dan *Abstract Syntax Tree*, mahasiswa diminta untuk melengkapi sebuah *Mini Compiler*.

### Tantangan Utama

Menambahkan dukungan untuk operator pangkat (`^`). Dalam hirarki matematika, operator pangkat memiliki prioritas lebih tinggi daripada operator perkalian (`*`) dan pembagian (`/`).

### 1. Definisi Node AST

Bagian ini mendefinisikan struktur data pohon untuk representasi kode.

In [1]:
class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class ParserError(Exception):
    pass

### 2. Implementasi MiniCompiler

Berikut adalah implementasi lengkap *MiniCompiler* yang telah dilengkapi pada bagian bertanda `# TUGAS`.

In [2]:
import re

class MiniCompiler:
    def __init__(self, source, env):
        # TUGAS 1: Memperbarui regex untuk mengenali simbol '^'
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env 
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        token = self._current
        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token and token.isalpha():
            if token not in self._env:
                raise ParserError(f"Semantic Error: Undefined variable '{token}'")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node
        raise ParserError(f"Unexpected token: {token}")

    # TUGAS 2: Implementasi fungsi power() untuk operator '^'
    def power(self):
        node = self.factor()
        while self._current == '^':
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.factor())
        return node

    # TUGAS 3: Memodifikasi term() untuk memanggil power()
    def term(self):
        node = self.power()
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power())
        return node

    def expr(self):
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def generate_tac(self, node):
        if isinstance(node, Num): return str(node.value)
        if isinstance(node, Var): return node.name
        
        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)
        
        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}")
        return temp_name

### 3. Uji Coba

Berikut adalah pengujian implementasi dengan ekspresi `a ^ 2 + b * c`.

In [3]:
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")
    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()
    
    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)
except Exception as e:
    print(f"Error: {e}")

Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


### 4. Pertanyaan Refleksi

#### Pertanyaan 1
**Mengapa fungsi `power()` harus dipanggil di dalam `term()`, bukan sebaliknya? Jelaskan kaitannya dengan *Operator Precedence*.**

**Jawaban:**

Fungsi `power()` dipanggil di dalam `term()` karena operator pangkat (`^`) memiliki tingkat prioritas (*precedence*) yang lebih tinggi daripada operator perkalian (`*`) dan pembagian (`/`). Dalam metode *recursive descent parsing*, fungsi yang menangani operator dengan prioritas lebih tinggi dipanggil pada level yang lebih dalam. Dengan memanggil `power()` dari dalam `term()`, ekspresi seperti `a ^ b * c` akan diurai menjadi `(a ^ b) * c`, sesuai dengan aturan matematika yang berlaku. Apabila susunan ini dibalik, maka prioritas operator tidak akan tercermin dengan benar dalam struktur AST yang dihasilkan.

#### Pertanyaan 2
**Apa yang terjadi pada fase **Analisis Semantik** jika variabel `z` digunakan dalam kode sumber tetapi tidak ada di `symbol_table`?**

**Jawaban:**

Pada fase analisis semantik, kompilator akan mendeteksi bahwa variabel `z` belum dideklarasikan atau didefinisikan. Dalam implementasi `MiniCompiler` di atas, ketika fungsi `factor()` menemukan token berupa identifier (huruf/alfabet), dilakukan pengecekan keberadaan variabel tersebut dalam `symbol_table` (`self._env`). Apabila variabel tidak ditemukan (`token not in self._env`), maka kompilator akan melemparkan *exception* `ParserError` dengan pesan *"Semantic Error: Undefined variable 'z'"*. Proses kompilasi akan dihentikan dan kode TAC tidak akan dihasilkan. Hal ini mencegah eksekusi program yang menggunakan variabel yang tidak memiliki nilai atau belum didefinisikan.

#### Pertanyaan 3
**Jelaskan mengapa dalam TAC, instruksi untuk `a ^ 2` harus muncul sebelum instruksi untuk `+`.**

**Jawaban:**

Dalam ekspresi `a ^ 2 + b * c`, operator pangkat (`^`) memiliki prioritas lebih tinggi daripada operator penjumlahan (`+`). Struktur AST yang dihasilkan oleh parser mencerminkan prioritas ini, sehingga node untuk `^` menjadi anak kiri dari node `+`. Ketika fungsi `generate_tac()` melakukan traversal secara *post-order* (rekursif), node anak kiri (`^`) akan diproses terlebih dahulu sebelum node `+`. Hal ini menghasilkan urutan kode tiga alamat sebagai berikut:

1. `t1 = a ^ 2`
2. `t2 = b * c`
3. `t3 = t1 + t2`

Urutan ini diperlukan karena nilai hasil dari `a ^ 2` harus tersedia terlebih dahulu sebelum operasi penjumlahan dapat dilakukan. Dengan demikian, urutan instruksi TAC mencerminkan ketergantungan data (*data dependency*) yang ada dalam ekspresi sumber.

---

**Hizkia Siallagan**  
231011400154 / 06TPLMP002